In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

##### Ofertas de Venda

In [2]:
def load_sales_data(path):
    pattern = os.path.join(path, "apartments_pl_*.csv")
    arquivos = glob.glob(pattern)
    
    dfs = []
    
    for arquivo in arquivos:
        df = pd.read_csv(arquivo)
        nome_arquivo = os.path.basename(arquivo)
        mes = nome_arquivo.replace(".csv", "")[-2:]
        
        df["mes"] = int(mes)
        dfs.append(df)
    
    df_final = pd.concat(dfs, ignore_index=True)
    
    return df_final

df = load_sales_data("../dados/bruto")

df.to_csv("../dados/bruto/apartments_pl.csv", index=False)

##### Ofertas de Aluguel

In [3]:
def load_rent_data(path):
    pattern = os.path.join(path, "apartments_rent_pl*.csv")
    arquivos = glob.glob(pattern)

    dfs = []

    for arquivo in arquivos:
        df = pd.read_csv(arquivo)
        nome_arquivo = os.path.basename(arquivo)
        mes = nome_arquivo.replace(".csv", "")[-2:]

        df["mes"] = int(mes)
        dfs.append(df)

    df_final = pd.concat(dfs, ignore_index=True)

    return df_final

df = load_rent_data("../dados/bruto")
df.to_csv("../dados/bruto/apartments_rent.csv", index=False)

### Limpeza de Dados

##### apartments_pl.csv

In [4]:
df = pd.read_csv("../dados/bruto/apartments_pl.csv")
df.head(5)

,id,city,type,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,...,ownership,buildingMaterial,condition,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom,price,mes
0,f8524536d4b09a0c8ccc0197ec9d7bde,szczecin,blockOfFlats,63.00,3.0,4.0,10.0,1980.0,53.378933,14.625296,...,condominium,concreteSlab,NaN,yes,yes,yes,no,yes,415000,8
1,accbe77d4b360fea9735f138a50608dd,szczecin,blockOfFlats,36.00,2.0,8.0,10.0,NaN,53.442692,14.559690,...,cooperative,concreteSlab,NaN,no,yes,yes,no,yes,395995,8
2,8373aa373dbc3fe7ca3b7434166b8766,szczecin,tenement,73.02,3.0,2.0,3.0,NaN,53.452222,14.553333,...,condominium,brick,NaN,no,no,no,no,no,565000,8
3,0a68cd14c44ec5140143ece75d739535,szczecin,tenement,87.60,3.0,2.0,3.0,NaN,53.435100,14.532900,...,condominium,brick,NaN,yes,yes,no,no,yes,640000,8
4,f66320e153c2441edc0fe293b54c8aeb,szczecin,blockOfFlats,66.00,3.0,1.0,3.0,NaN,53.410278,14.503611,...,condominium,NaN,NaN,no,no,no,no,no,759000,8


In [ ]:
# Colunas do DataFrame
print(f"Quantidade de Colunas: {len(df.columns)}\n")
print(f"Nomes das Colunas: {df.columns}\n")

# Informaçoes do DataFrame
df.info()

Quantidade de Colunas: 29

Nomes das Colunas: Index(['id', 'city', 'type', 'squareMeters', 'rooms', 'floor', 'floorCount',
       'buildYear', 'latitude', 'longitude', 'centreDistance', 'poiCount',
       'schoolDistance', 'clinicDistance', 'postOfficeDistance',
       'kindergartenDistance', 'restaurantDistance', 'collegeDistance',
       'pharmacyDistance', 'ownership', 'buildingMaterial', 'condition',
       'hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity',
       'hasStorageRoom', 'price', 'mes'],
      dtype='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195568 entries, 0 to 195567
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    195568 non-null  object 
 1   city                  195568 non-null  object 
 2   type                  153307 non-null  object 
 3   squareMeters          195568 non-null  float64
 4   rooms                 195568 

In [23]:
for coluna in df.columns:
    if df[coluna].isna().any():
        print(f"'{coluna}' possui valores nulos.")

print(f"\nQuantia de linhas com valores nulos: {df.isnull().any(axis=1).sum()}")

'type' possui valores nulos.
'floor' possui valores nulos.
'floorCount' possui valores nulos.
'buildYear' possui valores nulos.
'schoolDistance' possui valores nulos.
'clinicDistance' possui valores nulos.
'postOfficeDistance' possui valores nulos.
'kindergartenDistance' possui valores nulos.
'restaurantDistance' possui valores nulos.
'collegeDistance' possui valores nulos.
'pharmacyDistance' possui valores nulos.
'buildingMaterial' possui valores nulos.
'condition' possui valores nulos.
'hasElevator' possui valores nulos.

Quantia de linhas com valores nulos: 172141


##### Tratamento Coluna type

In [ ]:
# Quantia de cada valor dentro da coluna "type"
print(df["type"].value_counts(dropna=False))

# Porcentagem de cada valor dentro da coluna "type"
valores = df["type"].value_counts(normalize=True, dropna=False) * 100
print(valores.map('{:.2f}%'.format))

type
blockOfFlats         91368
NaN                  42261
apartmentBuilding    32507
tenement             29432
Name: count, dtype: int64
type
blockOfFlats         46.72%
NaN                  21.61%
apartmentBuilding    16.62%
tenement             15.05%
Name: proportion, dtype: object


###### Não é inteligente nesse caso normalizar os valores NaN como a moda "blockOfFlats", isso pode prejudicar a compreensão dos padrões de preço em relação ao tipo de imóvel mais tarde quando for criar o modelo de ML. Optarei por atribuir os campos que se encontram NaN como "Desconhecido". É a abordagem mais prática e cumprirá o objetivo esperado.

In [14]:
df["type"].fillna("Desconecido", inplace=True)
df["type"].value_counts(normalize=True, dropna=False) * 100

type
blockOfFlats         46.719300
Desconecido          21.609363
apartmentBuilding    16.621840
tenement             15.049497
Name: proportion, dtype: float64